In [11]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

In [12]:
dataset_path = "dataset/dataset-resized"

img_size = (224, 224)
batch_size = 32

In [13]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(train_data.class_indices)

Found 2024 images belonging to 6 classes.
Found 503 images belonging to 6 classes.
{'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}


In [14]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

In [15]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(6, activation="softmax")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │           7,686 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [16]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [17]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

Epoch 1/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 43s 569ms/step - accuracy: 0.5702 - loss: 1.1357 - val_accuracy: 0.6700 - val_loss: 0.8458
Epoch 2/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 34s 529ms/step - accuracy: 0.7678 - loss: 0.6477 - val_accuracy: 0.7217 - val_loss: 0.7072
Epoch 3/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 32s 498ms/step - accuracy: 0.8142 - loss: 0.5158 - val_accuracy: 0.7356 - val_loss: 0.6449
Epoch 4/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 31s 485ms/step - accuracy: 0.8454 - loss: 0.4529 - val_accuracy: 0.7435 - val_loss: 0.6433
Epoch 5/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 31s 480ms/step - accuracy: 0.8577 - loss: 0.3999 - val_accuracy: 0.7594 - val_loss: 0.6069
Epoch 6/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 31s 484ms/step - accuracy: 0.8745 - loss: 0.3613 - val_accuracy: 0.7634 - val_loss: 0.5990
Epoch 7/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 30s 473ms/step - accuracy: 0.8790 - loss: 0.3341 - val_accuracy: 0.7634 - val_loss: 0.6134
Epoch 8/15
64/64 ━━━━━━━━━━━━━━━━━━━━ 31s 486ms/step - accuracy: 0.8962 - loss: 0.3090 - val_accu

In [18]:
model.save("waste_model.keras")

In [19]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("waste_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully")

INFO:tensorflow:Assets written to: C:\Users\suman\AppData\Local\Temp\tmpr_vkapuq\assets


INFO:tensorflow:Assets written to: C:\Users\suman\AppData\Local\Temp\tmpr_vkapuq\assets


Saved artifact at 'C:\Users\suman\AppData\Local\Temp\tmpr_vkapuq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_313')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2428794631568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794633680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794633488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794633296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794632720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794632912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794633872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794632528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794631184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2428794634064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2

In [20]:
loss, accuracy = model.evaluate(val_data)

print(f"Validation Accuracy: {accuracy*100:.2f}%")

16/16 ━━━━━━━━━━━━━━━━━━━━ 6s 372ms/step - accuracy: 0.7674 - loss: 0.6140
Validation Accuracy: 76.74%


In [21]:
 import plotly.express as px

In [22]:
print(train_data.class_indices)

{'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}


In [23]:
pred = model.predict(val_data)

pred_classes = np.argmax(pred, axis=1)

unique, counts = np.unique(pred_classes, return_counts=True)

print(dict(zip(unique, counts)))

16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 452ms/step
{np.int64(0): np.int64(66), np.int64(1): np.int64(120), np.int64(2): np.int64(96), np.int64(3): np.int64(125), np.int64(4): np.int64(75), np.int64(5): np.int64(21)}
